# Information Flow: Information-Theoretic Layer Analysis

This notebook demonstrates IRTK's `information_flow` module:

1. **Layer entropy** – information content at each layer
2. **Mutual information** – how much input info is preserved
3. **Compression analysis** – effective dimensionality per layer
4. **Position flow** – information from source to target through layers
5. **Information bottleneck** – compression vs prediction trade-off

## Why This Matters

Information flow analysis measures how much information is transmitted, compressed, or created at each layer using entropy and mutual information. This reveals the model's information processing pipeline: which layers compress, which transform, and where bottlenecks occur.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.information_flow import (
    layer_entropy, mutual_information_estimate,
    compression_analysis, information_flow_by_position,
    information_bottleneck_curve,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
seqs = [jnp.array([0,1,2,3,4,5]), jnp.array([10,11,12,13])]
def metric_fn(logits): return float(logits[-1, 0])

In [ ]:
ent = layer_entropy(model, seqs)
print(f"Layer entropies: {ent['layer_entropies'].round(4)}")
print(f"Embedding entropy: {ent['embedding_entropy']:.4f}")
print(f"Trend: {ent['entropy_trend']}")

In [ ]:
mi = mutual_information_estimate(model, seqs, layer=0)
print(f"MI: {mi['mutual_information']:.4f}, normalized: {mi['normalized_mi']:.4f}")

In [ ]:
comp = compression_analysis(model, seqs)
print(f"Effective dims: {comp['effective_dimensions'].round(2)}")
print(f"Compression ratios: {comp['compression_ratios'].round(4)}")

In [ ]:
flow = information_flow_by_position(model, jnp.array([0,1,2,3,4,5]), source_pos=0, target_pos=5)
print(f"Layer influence: {flow['layer_influence'].round(4)}")
print(f"Peak layer: {flow['peak_layer']}")

In [ ]:
ib = information_bottleneck_curve(model, seqs, metric_fn)
print(f"Compression: {ib['compression'].round(4)}")
print(f"Prediction: {ib['prediction'].round(4)}")
print(f"Optimal layer: {ib['optimal_layer']}")